In [ ]:
import pandas as pd 
import numpy as np  
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from tqdm import tqdm  
import os 

In [ ]:
train_df = pd.read_parquet('train.parquet',
                           engine='fastparquet')
test_df = pd.read_parquet('test.parquet', engine='fastparquet')

def text_columns(df): # обьединение текста
    df['text1'] = df['title1'] + ' ' + df['description1'] + ' ' + df['subjectname1'] + ' ' + df['parentname1']
    df['text2'] = df['title2'] + ' ' + df['description2'] + ' ' + df['subjectname2'] + ' ' + df['parentname2']
    return df

train_df = text_columns(train_df)
test_df = text_columns(test_df)

tokenizer = AutoTokenizer.from_pretrained('cointegrated/rubert-tiny2')

In [ ]:
train_df
train_df.shape

In [ ]:
class PriceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained('cointegrated/rubert-tiny2')
        self.fc = nn.Sequential(
            nn.Linear(936, 384),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(384, 192),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(192, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2):
        output1 = self.bert(input_ids=input_ids1, attention_mask=attention_mask1)
        last_hidden1 = outputs1.last_hidden_state[:, 0, :] # берем cls токен
        output2 = self.bert(input_ids=input_ids2, attention_mask=attention_mask2)
        last_hidden2 = outputs2.last_hidden_state[:, 0, :]
        comb = torch.cat([last_hidden1, last_hidden2, torch.abs(last_hidden1 - last_hidden2)], dim=1) # сравниваем cls токены по модулю
        return self.fc(comb).squeeze()

In [ ]:
class PriceDataset(Dataset):
    def __init__(self, df, tokenizer, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text1 = str(self.df.iloc[idx]['text1'])
        text2 = str(self.df.iloc[idx]['text2'])
        encoding1 = self.tokenizer(
            text1,
            truncation=True,
            padding='max_length',
            max_length=392,
            return_tensors='pt'
        )
        encoding2 = self.tokenizer(
            text2,
            truncation=True,
            padding='max_length',
            max_length=392,
            return_tensors='pt'
        )
        result = {
            'input_ids1': encoding1['input_ids'].squeeze(),
            'attention_mask1': encoding1['attention_mask'].squeeze(),
            'input_ids2': encoding2['input_ids'].squeeze(), 
            'attention_mask2': encoding2['attention_mask'].squeeze(),
        }
        if self.is_train:
            result['label'] = torch.tensor(self.df.iloc[idx]['is_duplicates'], dtype=torch.float32)  # метка (для тренировочных данных)
        return result

In [ ]:
train_data, val_data = train_test_split(train_df, test_size=0.1,random_state=42)

train_dataset = PriceDataset(train_data, tokenizer, is_train=True)
val_dataset = PriceDataset(val_data, tokenizer, is_train=True)
test_dataset = PriceDataset(test_df, tokenizer, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=6, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=24, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=24, shuffle=False)

In [ ]:
device = torch.device('cuda')
model = PriceModel().to(device)

if os.path.exists('best_model.pth'):
    model.load_state_dict(torch.load('best_model.pth'))

optimizer = optim.Adam(model.parameters(), lr=3e-5)
criterion = nn.BCELoss()
best_val_loss = 999999999

for epoch in range(100):
    model.train()
    train_loss = 0 
    
    for batch in tqdm(train_loader, leave=False):
        input_ids1 = batch['input_ids1'].to(device)
        attention_mask1 = batch['attention_mask1'].to(device)
        input_ids2 = batch['input_ids2'].to(device)
        attention_mask2 = batch['attention_mask2'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad() 
        preds = model(input_ids1, attention_mask1, input_ids2, attention_mask2)
        loss = criterion(preds, labels) 
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    model.eval()
    val_loss = 0
    with torch.no_grad(): 
        for batch in val_loader: 
            input_ids1 = batch['input_ids1'].to(device)
            attention_mask1 = batch['attention_mask1'].to(device)
            input_ids2 = batch['input_ids2'].to(device)
            attention_mask2 = batch['attention_mask2'].to(device)
            labels = batch['label'].to(device)
            preds = model(input_ids1, attention_mask1, input_ids2, attention_mask2)
            loss = criterion(preds, labels)
            val_loss += loss.item()  
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"train Loss {avg_train_loss}, val Loss {avg_val_loss}")
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print("сохранена лучшая модель")
    else:
        print('ранняя остановка')
        break

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
predictions = []
with torch.no_grad(): 
    for batch in test_loader:
        input_ids1 = batch['input_ids1'].to(device)
        attention_mask1 = batch['attention_mask1'].to(device)
        input_ids2 = batch['input_ids2'].to(device)
        attention_mask2 = batch['attention_mask2'].to(device)
        preds = model(input_ids1, attention_mask1, input_ids2, attention_mask2)
        predictions.extend(preds.cpu().numpy())
predictions = np.array(predictions)
results = pd.DataFrame({
    'id': test_df['id'].values,
    'y_pred': predictions
})
results.to_csv('submission.csv', index=False)
print('все')